In [3]:
import json 

from datetime import datetime, timedelta
from operator import index
from CandleStream import CandleStream 
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
# from calender_utils import *
import utils
import dataset
# import strategy 
import numpy as np
from indicators import Indicators
# import mplfinance as mpf
import os

EXCHANGE = 'NSE'
START_DATE = datetime(2025, 12, 1, 9, 10)
END_DATE = datetime(2025, 12, 17, 15, 30)
START_TIME = datetime(2025, 10, 20, 9, 15)
END_TIME = datetime(2025, 10, 20, 12, 30)
INTERVAL = "5min"
INDEX = 'Nifty 50'
TRAIL = False
stream = CandleStream()
ORGINDEXDF = None
SECTORS_DATA = {}
STOCKS_DATA = {}

MAX_NUMBER_OF_STOCKS_PER_DAY = 5
RISK = 0.005
CAPITAL = 200000
RPT = CAPITAL * RISK
DAILYRISK = RPT * MAX_NUMBER_OF_STOCKS_PER_DAY//2
LESSVOLATILESTOCK =  ['TCS', 'HDFCBANK', 'SBIN']

TRADE_HISTORY = {}

###Strategy Params to have a bias
INITIALRANGE_IN_MINUTES = 10
MOMENTUM_IN_LAST_MINUTES = 20
###
ALLSECTORS = None
with open("index_stock.json", "r") as f:
    ALLSECTORS = json.load(f)


class CandleRange:
    def __init__(self, high, low):
        self.high = high
        self.low = low
        

def bias_structure(
    df: pd.DataFrame,
    window: int = 50,
    body_max_frac: float = 0.3,     # body <= 30% of range
    wick_min_frac: float = 0.5,     # long wick >= 50% of range
    top_band_frac: float = 0.2,     # body must close in top 30% (bullish pin)
    bottom_band_frac: float = 0.2,  # body must close in bottom 30% (bearish pin)
    vol_norm_window: int = 10,
    vol_clip: float = 3.0           # cap normalized volume to reduce outlier influence
    ):
    """
    Expects df with columns: open, high, low, close, volume.
    Returns df with pin classifications and a rolling bias score in [-1, +1].
    """
    df = df.copy()

    # Basic geometry
    rng = (df['high'] - df['low']).replace(0, np.nan)
    body = (df['close'] - df['open']).abs()
    upper_wick = df['high'] - np.maximum(df['open'], df['close'])
    lower_wick = np.minimum(df['open'], df['close']) - df['low']

    # Where is close within the range? (0=low, 1=high)
    close_pos = (df['close'] - df['low']) / rng

    # Thresholds
    small_body = (body / rng) <= body_max_frac
    long_upper = (upper_wick / rng) >= wick_min_frac
    long_lower = (lower_wick / rng) >= wick_min_frac

    close_near_top = close_pos >= (1 - top_band_frac)
    close_near_bottom = close_pos <= bottom_band_frac

    # Classifications
    bullish_pin = small_body & long_lower & close_near_top
    bearish_pin = small_body & long_upper & close_near_bottom

    # Percent body move (helps rank strength; sign matters)
    ret_body = (df['close'] - df['open']) / df['open']

    # Volume normalization
    vol_ma = df['volume'].rolling(vol_norm_window, min_periods=1).mean()
    vol_norm = (df['volume'] / vol_ma).clip(lower=0, upper=vol_clip)

    # Assign pressure based on pin and signed return magnitude
    # (You could also use wick/body ratio as weight. Here we combine sign and volume.)
    buy_pressure = np.where(bullish_pin, vol_norm * np.maximum(ret_body, 0), 0.0)
    sell_pressure = np.where(bearish_pin, vol_norm * np.maximum(-ret_body, 0), 0.0)

    df['bullish_pin'] = bullish_pin
    df['bearish_pin'] = bearish_pin
    df['vol_norm'] = vol_norm
    df['buy_p'] = buy_pressure
    df['sell_p'] = sell_pressure

    # Rolling aggregates
    buy_sum = pd.Series(buy_pressure, index=df.index).rolling(window, min_periods=5).sum()
    sell_sum = pd.Series(sell_pressure, index=df.index).rolling(window, min_periods=5).sum()

    # Bias metrics
    spread = buy_sum - sell_sum
    spread_total = (buy_sum + sell_sum).replace(0, np.nan)

    # Final bias score in [-1, +1]
    bias_score = (spread / spread_total).fillna(0).clip(-1, 1)

    df['pin_bias_score'] = bias_score
    df['pin_buy_sum'] = buy_sum
    df['pin_sell_sum'] = sell_sum



def data_bulk_loader():
    global ORGINDEXDF
    ORGINDEXDF = dataset.get_data(stream, 'NSE', INDEX, utils.get_token(EXCHANGE, INDEX), START_DATE, END_DATE, INTERVAL)
    ORGINDEXDF = Indicators.ema(ORGINDEXDF, 9)
    ORGINDEXDF = Indicators.ema(ORGINDEXDF, 15)
    
    
    for sector, stocks in ALLSECTORS.items():
        sectordf = dataset.get_data(stream, 'NSE', sector, utils.get_token_for_index(EXCHANGE, sector), START_DATE, END_DATE, INTERVAL)
        if sectordf is None:
            print(f"Data not found for sector: {sector}")
        else:
            sectordf = Indicators.ema(sectordf, 9)
            sectordf = Indicators.ema(sectordf, 15)
            SECTORS_DATA[sector] = sectordf

        for symbol in stocks:
            stockdf = dataset.get_data(stream, 'NSE', symbol, utils.get_token(EXCHANGE, symbol + '-EQ'), START_DATE, END_DATE, INTERVAL)

            if stockdf is None:
                print(f"Data not found for stock: {symbol}")
            else:
                stockdf = Indicators.ema(stockdf, 9)
                stockdf = Indicators.ema(stockdf, 15)
                STOCKS_DATA[symbol] = stockdf



[I 251219 14:41:01 smartConnect:121] in pool


In [4]:

data_bulk_loader()

NSE Nifty 50 99926000 2025-12-31 23:59:59
2025-12-31 23:59:59
Syncing for Nifty 50 from 2025-12-01 00:00:00 to 2025-12-31 23:59:59
NSE NIFTY IT 99926008 2025-12-31 23:59:59
2025-12-31 23:59:59
Syncing for NIFTY IT from 2025-12-01 00:00:00 to 2025-12-31 23:59:59
NSE LTIM 17818 2025-12-31 23:59:59
2025-12-31 23:59:59
Syncing for LTIM from 2025-12-01 00:00:00 to 2025-12-31 23:59:59
NSE TECHM 13538 2025-12-31 23:59:59
2025-12-31 23:59:59
Syncing for TECHM from 2025-12-01 00:00:00 to 2025-12-31 23:59:59
NSE WIPRO 3787 2025-12-31 23:59:59
2025-12-31 23:59:59
Syncing for WIPRO from 2025-12-01 00:00:00 to 2025-12-31 23:59:59
NSE OFSS 10738 2025-12-31 23:59:59
2025-12-31 23:59:59
Syncing for OFSS from 2025-12-01 00:00:00 to 2025-12-31 23:59:59
NSE TCS 11536 2025-12-31 23:59:59
2025-12-31 23:59:59
Syncing for TCS from 2025-12-01 00:00:00 to 2025-12-31 23:59:59
NSE HCLTECH 7229 2025-12-31 23:59:59
2025-12-31 23:59:59
Syncing for HCLTECH from 2025-12-01 00:00:00 to 2025-12-31 23:59:59
NSE INFY 159

In [ ]:
# START_TIME = datetime(2025, 10, 20, 10, 10)
# END_TIME = datetime(2025, 10, 20, 11, 30)
TRADE_HISTORY = {}
TOPSECTORS = 2

In [5]:
TRADE_HISTORY = {}
TOPSECTORS = 2
def filter_data_by_date(df, date):
    filtered_df = df[df['date'] == date.date()]
    filtered_df.reset_index(inplace=True, drop =True)
    return filtered_df

def filter_data_by_time(df, start_time, end_time):
    filtered_df = df[(df['timestamp'].dt.time >= start_time) & (df['timestamp'].dt.time <= end_time)]
    filtered_df.reset_index(inplace=True, drop =True)
    return filtered_df

def get_minor_data(df, date, starttime, end_time):
    df = filter_data_by_date(df, date)
    if df.empty:
        return 
    df = filter_data_by_time(df, starttime, end_time)
    return df


import mplfinance as mpf
def save_image(symbol, date, entry, sl, target):
        df = STOCKS_DATA.get(symbol)
        df = filter_data_by_date(df, date)
        foldername = "-".join(str(date).split(':'))
        filename = symbol
        folderpath = f"samplesimages/{foldername}" 
        filepath = f"samplesimages/{foldername}/{filename}.jpg"
        os.makedirs(folderpath, exist_ok=True)
        df['time'] = pd.to_datetime(df['timestamp'])
        df.set_index('time', inplace=True)
        entry_m = pd.Series(entry, index=df.index)
        ema_plots = []
        ema_plots.append(mpf.make_addplot(entry_m, color='green', linestyle='-', width=3))
        
        sl_m = pd.Series(sl, index=df.index)
        ema_plots.append(mpf.make_addplot(sl_m, color='red', linestyle='-', width=5))

        mpf.plot(df, type='candle', style='charles', title=symbol,
                ylabel='Price',
                figsize=(32, 16),
                volume=True,
                addplot=ema_plots,
                savefig=dict(fname=filepath, dpi=100))
        
import math
def Intraday_breakouts(df, window):
    df['resistance'] = np.nan
    df['max'] = df['high'].cummax()
    df['isheighest'] = df['max'] == df['high']

    for i in range(len(df)):
        if i == len(df)-window:
            break

        if i == 0:
            right = df.iloc[i+1:i+window+1]
            curr = df.iloc[i]
            if curr['high'] > right['high'].max() and curr['isheighest']:
                df.loc[i, 'resistance'] = df.iloc[i]['high']
        
        else:
            left = df.iloc[max(i-window,0) : i+1]
            right = df.iloc[i : i+window+1]
            curr = df.iloc[i]
            if curr['high'] >= left['high'].max() and curr['high'] >= right['high'].max() and curr['isheighest']:
                df.loc[i, 'resistance'] = df.iloc[i]['high']

    df['resistance'] = df['resistance'].ffill()
    df['resistance'] = df['resistance'].shift(window)
    df = df.drop(columns = ['isheighest'])
    
    return df 


def fit_strategy(df, side, date, t):
    if side == 'bullish':
            df = Intraday_breakouts(df, 3)
            if df.iloc[-1]['breakout']:
                entry =  df.iloc[-1]['close']
                sl = df.iloc[-4:]['low'].min()
                target = entry + 3 * (entry - sl)
                quantity = RPT//math.ceil(abs(sl - entry)/5)
                if quantity == 1:
                    return
                return {'entry' : entry, 'sl' : sl, 'target' : target, 'date' : date, 'time' : t, 'status' : True, 'side' : side, 'quantity' : quantity}
        
    if side == 'bearish':
            df = Intraday_breakdowns(df, 3)
            if df.iloc[-1]['breakdown']:
                entry =  df.iloc[-1]['close']
                sl = df.iloc[-3:]['high'].max()
                target = entry - 3 * (sl - entry)
                quantity = RPT//math.ceil(abs(sl - entry)/5)
                if quantity == 1:
                    return
                return {'entry' : entry, 'sl' : sl, 'target' : target, 'date' : date, 'time' : t, 'status' : True, 'side': side, 'quantity' : quantity }



def manage_trades(stock, is_trade, curr_t, date):
    update_profit_state(curr_t)
    netrpofit = 0
    totalnumberoftrades = 0
    for stock, trade in TRADE_HISTORY.items():
        netrpofit += trade.get('profit', 0)
        totalnumberoftrades += 1

    save_image(stock, date, is_trade['entry'], is_trade['sl'], None)
    if totalnumberoftrades > MAX_NUMBER_OF_STOCKS_PER_DAY:
        return 
    
    if netrpofit < -1 * DAILYRISK:
        return 

    print(f"At time {curr_t}, Total trades: {totalnumberoftrades}, Net profit: {netrpofit}")
    if totalnumberoftrades == 0 :
        print(f"Placing {totalnumberoftrades + 1}rd trade at time {curr_t} as net profit {netrpofit} > RPT {RPT//2} in ---- {stock}")
        TRADE_HISTORY[stock] = is_trade

    elif totalnumberoftrades == 1 :
        if netrpofit > RPT//2:
            print(f"Placing {totalnumberoftrades + 1}nd trade at time {curr_t} as net profit {netrpofit} > RPT {RPT//2} in ---- {stock}")
            TRADE_HISTORY[stock] = is_trade

    elif totalnumberoftrades == 2:
        if netrpofit > RPT:
            print(f"Placing {totalnumberoftrades + 1}rd trade at time {curr_t} as net profit {netrpofit} > RPT {RPT} in ---- {stock}")
            TRADE_HISTORY[stock] = is_trade

    elif totalnumberoftrades == 3:
        if netrpofit > RPT * 2:
            print(f"Placing {totalnumberoftrades + 1}rd trade at time {curr_t} as net profit {netrpofit} > RPT {RPT * 2} in ---- {stock}")
            TRADE_HISTORY[stock] = is_trade

    elif totalnumberoftrades >= 4:
        if netrpofit > RPT * 3:
            print(f"Placing {totalnumberoftrades + 1}rd trade at time {curr_t} as net profit {netrpofit} > RPT {RPT * 3} in ---- {stock}")
            TRADE_HISTORY[stock] = is_trade
    

def update_profit_state(current_time):
    global TRADE_HISTORY
    hisotry_temp = TRADE_HISTORY.copy()

    for stock, trade in TRADE_HISTORY.items():
        stockdf = STOCKS_DATA.get(stock)
        if stockdf is None:
            print(f"This is not Possible as this trade already exists for stock : {stock} with trade \n {trade}")

        else:
            stockdf = get_minor_data(stockdf, trade['date'], START_TIME.time(), current_time)
            currprice = stockdf.iloc[-1]['close']
            if trade['side'] == 'bullish' and trade['status'] == True:
                if currprice < trade['sl']:
                    profit = (trade['sl'] - trade['entry']) * trade['quantity']
                    hisotry_temp[stock]['status'] = False
                    hisotry_temp[stock]['exit_price'] = trade['sl']
                    hisotry_temp[stock]['notes'] = 'sl hit'
                    hisotry_temp[stock]['profit'] = profit
                elif currprice > trade['target'] : 
                    profit = (trade['target'] - trade['entry']) * trade['quantity']
                    hisotry_temp[stock]['status'] = False
                    hisotry_temp[stock]['exit_price'] = trade['target']
                    hisotry_temp[stock]['notes'] = 'target achieved'
                    hisotry_temp[stock]['profit'] = profit
                else:
                    profit = (currprice - trade['entry']) * trade['quantity']
                    hisotry_temp[stock]['profit'] = profit
                

                if TRAIL and hisotry_temp[stock]['status'] == True and trade['sl'] != trade['entry']:
                    new_sl = trade['entry']
                    if currprice - trade['entry'] > 2*(trade['entry'] - trade['sl']):
                        hisotry_temp[stock]['sl'] = new_sl


            if trade['side'] == 'bearish' and trade['status'] == True:
                if currprice > trade['sl']:
                    profit = (trade['entry'] - trade['sl']) * trade['quantity']
                    hisotry_temp[stock]['status'] = False
                    hisotry_temp[stock]['exit_price'] = trade['sl']
                    hisotry_temp[stock]['notes'] = 'sl hit' 
                    hisotry_temp[stock]['profit'] = profit
                elif currprice < trade['target'] :
                    profit = (trade['entry'] - trade['target']) * trade['quantity']
                    hisotry_temp[stock]['status'] = False
                    hisotry_temp[stock]['exit_price'] = trade['target']
                    hisotry_temp[stock]['notes'] = 'target achieved'
                    hisotry_temp[stock]['profit'] = profit
                else:
                    profit = (trade['entry'] - currprice) * trade['quantity']
                    hisotry_temp[stock]['profit'] = profit 

                if TRAIL and hisotry_temp[stock]['status'] == True and trade['sl'] != trade['entry']:
                    new_sl = trade['entry']
                    if trade['entry'] - currprice > 2*(trade['sl'] - trade['entry']):
                        hisotry_temp[stock]['sl'] = new_sl

    TRADE_HISTORY = hisotry_temp.copy()

def trend(df, endtime, first_15_mins):
    df['bearcandle'] = (df['close'] < df['ema_9']) & (df['close'] < df['ema_15']) & (df['close'] < np.minimum(df['open'].shift(1), df['close'].shift(1))) 
    df['bullcandle'] = (df['close'] > df['ema_9']) & (df['close'] > df['ema_15']) & (df['close'] > np.maximum(df['open'].shift(1), df['close'].shift(1))) 
    df['side'] = np.where(df['bearcandle'] , 'bearish',
                          np.where(df['bullcandle'], 'bullish', ''))
    
    if df.iloc[-1]['time'] == endtime.time():
        if df.iloc[-1]['close'] > first_15_mins['high'].max() and (df.iloc[-1]['side'] == 'bullish' or df.iloc[-2]['side'] == 'bullish' ):
            return "bullish"
        
        if df.iloc[-1]['close'] < first_15_mins['low'].min() and (df.iloc[-1]['side'] == 'bearish' or df.iloc[-2]['side'] == 'bearish'):
            return "bearish"        


def get_nifty_view(date, starttime, endtime):
    df = ORGINDEXDF.copy()
    df = filter_data_by_date(df, date)
    if df is None or len(df) == 0:
        return 
    df = filter_data_by_time(df, starttime.time(), endtime.time())
    df.reset_index(inplace = True, drop = True)
    first_15_mins = filter_data_by_time(df, starttime.time(), datetime(2025, 1,1, 9, 30).time())
    return trend(df, endtime, first_15_mins)
    

def get_change(symbol, date, starttime, endtime, side, what):
    df = None
    if what == 'sector':
        df = SECTORS_DATA.get(symbol)
    if what == 'stock':
        df = STOCKS_DATA.get(symbol)
    df = filter_data_by_date(df, date)
    if df is None or len(df) == 0:
        return None
    df = filter_data_by_time(df, starttime.time(), endtime.time())
    first_15_mins = filter_data_by_time(df, starttime.time(), datetime(2025, 1,1, 9, 30).time())
    if side == 'bullish' :
        daylow = df['low'].min()
        currprice = df.iloc[-1]['close']
        change = (currprice - daylow)/daylow * 100
        return round(change , 2), trend(df, endtime, first_15_mins)
    
    if side == 'bearish' :
        dayhigh = df['high'].max()
        currprice = df.iloc[-1]['close']
        change = (currprice - dayhigh)/dayhigh * 100
        return round(change , 2), trend(df, endtime, first_15_mins)
    

def get_stock_view(stock, date, starttime, endtime, side):
    df = SECTORS_DATA.get(stock)
    df = filter_data_by_date(df, date)
    if df is None or len(df) == 0:
        return 
    df = filter_data_by_time(df, starttime.time(), endtime.time())
    df.reset_index(inplace = True, drop = True)
    first_15_mins = filter_data_by_time(df, starttime.time(), datetime(2025, 1,1, 9, 30).time())
    return trend(df, endtime, first_15_mins)
    
def get_change_threshold(stock, side):
    if stock in LESSVOLATILESTOCK:
        if side == 'bullish':
            return 1
        else:
            return -1
    
    if side == 'bullish':
        return 2
    else:
        return -2
    
def Intraday_breakouts(df, window):
    df.reset_index(inplace=True, drop =True)
    df['resistance'] = np.nan
    df['max'] = df['high'].cummax()
    df['isheighest'] = df['max'] == df['high']

    for i in range(len(df)):
        if i == len(df)-window:
            break

        if i == 0:
            right = df.iloc[i+1:i+window+1]
            curr = df.iloc[i]
            if curr['high'] > right['high'].max() and curr['isheighest']:
                df.loc[i, 'resistance'] = df.iloc[i]['high']
        
        else:
            left = df.iloc[max(i-window,0) : i+1]
            right = df.iloc[i : i+window+1]
            curr = df.iloc[i]
            if curr['high'] >= left['high'].max() and curr['high'] >= right['high'].max() and curr['isheighest']:
                df.loc[i, 'resistance'] = df.iloc[i]['high']

    df['resistance'] = df['resistance'].ffill()
    df['resistance'] = df['resistance'].shift(window)
    df = df.drop(columns = ['isheighest'])
    df['breakout'] = (df['close'] > df['resistance']) & (df['high'].cummax().shift(1) == df['resistance'].shift(1))
    return df 

def Intraday_breakdowns(df, window):
    df.reset_index(inplace=True, drop =True)
    df['support'] = np.nan
    df['min'] = df['low'].cummin()
    df['islowest'] = df['min'] == df['low']

    for i in range(len(df)):
        if i == len(df)-window:
            break

        if i == 0:
            right = df.iloc[i+1:i+window+1]
            curr = df.iloc[i]
            if curr['low'] < right['low'].min() and curr['islowest']:
                df.loc[i, 'support'] = df.iloc[i]['low']
        
        else:
            left = df.iloc[max(i-window,0) : i+1]
            right = df.iloc[i : i+window+1]
            curr = df.iloc[i]
            if curr['low'] <= left['low'].min() and curr['low'] <= right['low'].min() and curr['islowest']:
                df.loc[i, 'support'] = df.iloc[i]['low']

    df['support'] = df['support'].ffill()
    df['support'] = df['support'].shift(window)
    df = df.drop(columns = ['islowest'])
    df['breakdown'] = (df['close'] < df['support']) & (df['low'].cummin().shift(1) == df['support'].shift(1))
    return df 


def day_trading(date):
    '''This will do day trading for all stocks in index_stock.json for a given date'''
    print("Starting day trading for date:", date)
    timer = datetime(2025, 1, 1, 10, 0)
    if ORGINDEXDF is None:
        print(f"XXXXXXXXXXXXXXXXXXXXXXXXXXX           \n No index data for {date}: may be some issue check again")
    indexdf = filter_data_by_date(ORGINDEXDF.copy(), date)
    if indexdf is None or indexdf.empty:
        print(f"XXXXXXXXXXXXXXXXXXXXXXXXXXX           \n No index data for {date}: may be a holiday check again")
        return 
    
    trading_time_range = pd.date_range("10:00", "15:30", freq="5min")
    for timer in trading_time_range:
        niftyview = get_nifty_view(date, START_TIME, timer) 
        sectorialview = {}
        if niftyview is not None:
            for sector in ALLSECTORS:
                sectorchange = get_change(sector, date, START_TIME, timer, niftyview, 'sector')
                if sectorchange is not None:
                    sectorialview[sector] = sectorchange
            if niftyview == 'bearish':
                sectorialview = sorted(sectorialview.items(), key=lambda x: x[1][0])
                sectorialview = sectorialview[:TOPSECTORS]
                for sector, (change, trend) in sectorialview:
                    if trend != niftyview:
                        continue    
                    print(f"niftyview -> {niftyview}  :     {sector}  ->  {trend}")
                    allstocksview = {}
                    for stock in ALLSECTORS.get(sector):
                        stockchange = get_change(stock, date, START_TIME, timer, niftyview, 'stock')
                        if stockchange is not None:
                            allstocksview[stock] = stockchange
                    allstocksview = sorted(allstocksview.items(), key=lambda x: x[1][0])

                    for stock, (change, trend) in allstocksview:
                        if TRADE_HISTORY.get(stock) is not None:
                            continue
                        if trend != niftyview:
                            continue
                        print(f"niftyview -> {niftyview}     and      {stock}  ->  {trend}")
                        if change < get_change_threshold(stock, trend):
              
                            df = STOCKS_DATA.get(stock)
                            df = filter_data_by_date(df, date)
                            if df is not None and  len(df) > 0:
                                df = filter_data_by_time(df, START_TIME.time(), timer.time())
                         
                                print(change, get_change_threshold(stock, trend))
                                istrade = fit_strategy(df, trend, date, timer)
                                
                                if istrade is None:
                                    continue
                                else:
                                    print(df)
                                    manage_trades(stock, istrade, timer.time(), date)

        # print(timer, ' -> ' ,niftyview, ' :  ' ,  sectorialview)
        # update_profit_state(t)
    return 


In [6]:

day_trading(END_DATE)
print(TRADE_HISTORY)

Starting day trading for date: 2025-12-17 15:30:00
niftyview -> bearish  :     NIFTY METAL  ->  bearish
niftyview -> bearish     and      VEDL  ->  bearish
-2.12 -2
niftyview -> bearish     and      ADANIENT  ->  bearish
niftyview -> bearish     and      APLAPOLLO  ->  bearish
niftyview -> bearish  :     NIFTY PVT BANK  ->  bearish
niftyview -> bearish     and      HDFCBANK  ->  bearish
niftyview -> bearish  :     FINNIFTY  ->  bearish
niftyview -> bearish     and      SBILIFE  ->  bearish
niftyview -> bearish     and      SBICARD  ->  bearish
niftyview -> bearish     and      HDFCLIFE  ->  bearish
niftyview -> bearish     and      HDFCAMC  ->  bearish
niftyview -> bearish     and      MUTHOOTFIN  ->  bearish
niftyview -> bearish     and      JIOFIN  ->  bearish
niftyview -> bearish  :     FINNIFTY  ->  bearish
niftyview -> bearish     and      SBILIFE  ->  bearish
niftyview -> bearish     and      SBICARD  ->  bearish
niftyview -> bearish     and      HDFCAMC  ->  bearish
niftyview ->

In [ ]:
TRADE_HISTORY

In [ ]:
ALLSECTORS.keys()

In [ ]:


END_DATE = datetime(2025, 12, 17, 15, 30)
stock = 'SBILIFE'
df = dataset.get_data(stream, 'NSE', stock, utils.get_token(EXCHANGE, stock + '-EQ'), START_DATE, END_DATE, INTERVAL)
df = Indicators.ema(df, 9)
df = Indicators.ema(df, 15)

df = filter_data_by_date(df, END_DATE)
# df = df.iloc[4:,:]
df.reset_index(inplace=True, drop =True)
if df is not None and  len(df) > 0:
    df = Intraday_breakdowns(df, 3)
    print(df.head(50))
    print(df[df['breakdown']])
